# 03 - Concept Bottleneck, Sanity Checks and Advanced Attribution Audit

This notebook consolidates **the concept-bottleneck audit and the saliency sanity audit**.

Technical scope:

1. **Concept Bottleneck evaluation**  
   Compare the direct ResNet classifier with a model whose class prediction is mediated by AwA2 semantic concepts.
2. **Explanation robustness audit**  
   Compare vanilla input-gradient saliency with SmoothGrad, Occlusion Sensitivity and randomized-model saliency.

The notebook is restricted to experimental analysis. Static blog/report generation is intentionally excluded from this notebook.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
import csv
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "scripts").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the project root containing src/ and scripts/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST = PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"
METADATA_ROOT = PROJECT_ROOT / "data" / "AWA2"
CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"

print("project_root:", PROJECT_ROOT)
print("manifest:", MANIFEST, "exists=", MANIFEST.exists())
print("checkpoint:", CHECKPOINT, "exists=", CHECKPOINT.exists())

RUN_CBM = True
RUN_SANITY = True
RUN_ADVANCED_ATTRIBUTION_AUDIT = True


project_root: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI
manifest: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/data/AWA2_subset_background20/awa2_manifest_subset.csv exists= True
checkpoint: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/checkpoints/best_resnet50_awa2.pt exists= True


## Explainability Scope

This notebook evaluates two explanation paradigms that go beyond raw heatmaps:

- **Concept Bottleneck Models** make concepts part of the prediction path. The class head receives predicted semantic attributes rather than arbitrary hidden features.
- **Sanity checks** evaluate whether saliency maps are stable, perturbation-consistent and dependent on learned model parameters.

The focus is on explanation reliability. Accuracy remains important, but it is interpreted together with concept quality, agreement with the baseline and saliency robustness metrics.


## 1. Concept Bottleneck Model

A Concept Bottleneck Model changes the prediction pipeline:

```text
image -> predicted concepts -> class
```

Unlike TCAV, this is not only post-hoc. Concepts become part of the model's decision path. This makes predictions easier to inspect and intervene on, but it may reduce accuracy because the model can no longer use arbitrary hidden features.

In this project, the concept targets come from AwA2 class-level attributes. That means every image of a class receives the same concept target. This is useful for an audit, but weaker than image-level concept annotation.

The notebook also inspects a **concept confusion matrix**. For every bottleneck concept, the script counts true positives, false positives, false negatives and true negatives after thresholding predicted concept probabilities. This identifies which semantic concepts are reliable and which ones are systematically confused.


In [7]:
if RUN_CBM:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'train_cbm.py'),
        '--manifest', str(MANIFEST),
        '--metadata-root', str(METADATA_ROOT),
        '--backbone-checkpoint', str(CHECKPOINT),
        '--checkpoint-output', str(PROJECT_ROOT / 'outputs' / 'checkpoints' / 'phase8_cbm_notebook.pt'),
        '--history-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_cbm_history_notebook.csv'),
        '--summary-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_cbm_summary_notebook.csv'),
        '--concept-metrics-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_metrics_notebook.csv'),
        '--concept-confusion-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_confusion_matrix_notebook.csv'),
        '--predictions-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_cbm_predictions_notebook.csv'),
        '--intervention-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_interventions_notebook.csv'),
        '--training-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_cbm_training_notebook.png'),
        '--summary-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_cbm_summary_notebook.png'),
        '--concept-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_prediction_metrics_notebook.png'),
        '--concept-confusion-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_confusion_matrix_notebook.png'),
        '--intervention-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_interventions_notebook.png'),
        '--top-concepts', '20',
        '--epochs', '5',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('CBM run skipped')

for csv_path in [
    PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_cbm_summary_notebook.csv',
    PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_metrics_notebook.csv',
    PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_confusion_matrix_notebook.csv',
]:
    print('\n', csv_path)
    if csv_path.exists():
        with csv_path.open('r', newline='', encoding='utf-8') as handle:
            rows = list(csv.DictReader(handle))
        print('rows:', len(rows))
        for row in rows[:8]:
            print(row)
    else:
        print('missing')

for figure in [
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_cbm_summary_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_cbm_training_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_prediction_metrics_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_confusion_matrix_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase8_concept_interventions_notebook.png',
]:
    print(figure)
    if figure.exists():
        display(Image(filename=str(figure)))
    else:
        print('missing')


CBM run skipped

 /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase8_cbm_summary_notebook.csv
missing

 /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase8_concept_metrics_notebook.csv
missing

 /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase8_concept_confusion_matrix_notebook.csv
missing
/Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/figures/phase8_cbm_summary_notebook.png
missing
/Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/figures/phase8_cbm_training_notebook.png
missing
/Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/figures/phase8_concept_prediction_metrics_notebook.png
missing
/Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/figures/phase8_concept_confusion_matrix_notebook.png
missing
/Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/figures/phase8_concept_interventions

### Concept Bottleneck Result Interpretation

The Concept Bottleneck Model should be evaluated with four quantities:

- **CBM accuracy**: predictive performance after forcing the decision through concepts.
- **Baseline agreement**: fraction of predictions matching the direct ResNet classifier.
- **Concept prediction error**: quality of the intermediate semantic representation.
- **Concept confusion matrix**: per-concept TP/FP/FN/TN counts after thresholding predicted concept probabilities.

A competitive CBM indicates that selected concepts preserve relevant information. A performance drop indicates that the direct model uses information outside the concept vocabulary or that concept targets are too coarse. The confusion matrix adds a more local diagnostic: it shows whether the bottleneck fails because some concepts are missed, over-predicted or correctly suppressed.


In [8]:
summary_path = PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_cbm_summary_notebook.csv'
confusion_path = PROJECT_ROOT / 'outputs' / 'reports' / 'phase8_concept_confusion_matrix_notebook.csv'
if summary_path.exists():
    with summary_path.open('r', newline='', encoding='utf-8') as handle:
        summary = next(csv.DictReader(handle))
    print('CBM accuracy:', float(summary['test_class_acc']))
    print('Baseline accuracy:', float(summary['baseline_accuracy']))
    print('CBM / baseline agreement:', float(summary['cbm_baseline_agreement']))
    print('Concept MAE:', float(summary['test_concept_mae']))
    print('Concept binary accuracy:', float(summary['test_concept_binary_acc']))
else:
    print('missing:', summary_path)

if confusion_path.exists():
    with confusion_path.open('r', newline='', encoding='utf-8') as handle:
        confusion_rows = list(csv.DictReader(handle))
    confusion_rows = sorted(confusion_rows, key=lambda row: float(row['f1']))
    print('\nLowest-F1 bottleneck concepts:')
    for row in confusion_rows[:10]:
        print({
            'concept': row['concept'],
            'f1': round(float(row['f1']), 4),
            'precision': round(float(row['precision']), 4),
            'recall': round(float(row['recall']), 4),
            'tp': int(row['true_positive']),
            'fp': int(row['false_positive']),
            'fn': int(row['false_negative']),
            'tn': int(row['true_negative']),
        })
else:
    print('missing:', confusion_path)


missing: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase8_cbm_summary_notebook.csv
missing: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase8_concept_confusion_matrix_notebook.csv


### Interpretation checkpoint

The current CBM result is strong but not perfect: it is less accurate than the direct ResNet, yet still competitive. This is the trade-off the project should emphasize: interpretability improves, but the concept bottleneck can lose information and concept predictions can be noisy.


## 2. Robustness and Sanity Checks

the saliency sanity audit evaluates saliency reliability through complementary diagnostics.

Comparisons:

- **SmoothGrad**: averages input gradients over noisy copies of the same image to estimate noise stability.
- **Occlusion Sensitivity**: measures target-probability drop after replacing local patches with a blurred baseline.
- **Randomized-model gradients**: recomputes saliency after reinitializing model weights to test model dependence.

A reliable explanation should show controlled stability under SmoothGrad, partial agreement with occlusion-based evidence and low similarity to randomized-model saliency.


In [9]:
if RUN_SANITY:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'audits' / 'run_saliency_sanity_audit.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint', str(CHECKPOINT),
        '--num-examples', '4',
        '--metrics-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase9_explainability_audit_notebook.csv'),
        '--figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase9_explainability_audit_notebook.png'),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('Saliency sanity audit skipped')

csv_path = PROJECT_ROOT / 'outputs' / 'reports' / 'phase9_explainability_audit_notebook.csv'
if csv_path.exists():
    with csv_path.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        print(row)

figure = PROJECT_ROOT / 'outputs' / 'figures' / 'phase9_explainability_audit_notebook.png'
if figure.exists():
    display(Image(filename=str(figure)))


Saliency sanity audit skipped


### Sanity-Check Result Interpretation

The randomized-model comparison is a model-dependence test. Low similarity between trained-model and randomized-model saliency is desirable, because it indicates that the explanation depends on learned parameters.

SmoothGrad and Occlusion comparisons evaluate stability and perturbation consistency. Moderate or low agreement indicates that the exact highlighted pixels are method-dependent and should not be treated as a definitive causal explanation.


In [10]:
audit_path = PROJECT_ROOT / 'outputs' / 'reports' / 'phase9_explainability_audit_notebook.csv'
if audit_path.exists():
    with audit_path.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    summary = next((row for row in rows if row['index'] == 'summary_mean'), None)
    if summary is not None:
        print('Mean vanilla vs SmoothGrad IoU:', summary['vanilla_vs_smoothgrad_iou_top20'])
        print('Mean vanilla vs SmoothGrad Spearman:', summary['vanilla_vs_smoothgrad_spearman'])
        print('Mean vanilla vs Occlusion IoU:', summary['vanilla_vs_occlusion_iou_top20'])
        print('Mean vanilla vs Occlusion Spearman:', summary['vanilla_vs_occlusion_spearman'])
        print('Mean vanilla vs randomized IoU:', summary['vanilla_vs_randomized_iou_top20'])
        print('Mean vanilla vs randomized Spearman:', summary['vanilla_vs_randomized_spearman'])
    else:
        print('summary row not found')
else:
    print('missing:', audit_path)


missing: /Users/alinafogar/Desktop/Codici/DL-Project/Deep_Learning_XAI/outputs/reports/phase9_explainability_audit_notebook.csv



Low similarity with the randomized model indicates that saliency depends on learned model parameters rather than only on image edges or preprocessing artifacts. Moderate agreement between vanilla gradients, SmoothGrad and Occlusion Sensitivity indicates method dependence; the saliency map should not be treated as a definitive causal mask.
